In [60]:
import json 
from pathlib import Path

In [61]:
unoffitial_path = Path("./annotations_unofficial.json")
unoffitial_path.exists()

True

In [62]:
with open(unoffitial_path, "r") as f:
  unoffitial_meta = json.load(f)

In [63]:
type(unoffitial_meta), unoffitial_meta.keys()

(dict,
 dict_keys(['categories', 'images', 'info', 'scene_categories', 'scene_annotations', 'annotations']))

In [64]:
unoffitial_path = Path("./annotations_unofficial.json")
unoffitial_path.exists()

True

In [65]:
annotation_path = Path("./annotations.json")
annotation_path.exists()

True

In [66]:
with open(annotation_path, "r") as f:
  annotaiton_meta = json.load(f)

In [67]:
annotaiton_meta.keys()

dict_keys(['info', 'images', 'annotations', 'scene_annotations', 'licenses', 'categories', 'scene_categories'])

In [68]:
len(annotaiton_meta["images"]), annotaiton_meta["images"][0]

(1500,
 {'id': 0,
  'width': 1537,
  'height': 2049,
  'file_name': 'batch_1/000006.jpg',
  'license': None,
  'flickr_url': 'https://img.tacodataset.org/img/33978196618.jpg',
  'coco_url': None,
  'date_captured': None,
  'flickr_640_url': 'https://img.tacodataset.org/thumb/33978196618_640.jpg'})

In [69]:
VAL_RATIO = 0.2
assert 0 < VAL_RATIO < 1

YOLO_DIR = annotation_path.parent / "taco_yolo"
YOLO_IMAGE_DIR = YOLO_DIR / "images"
YOLO_LABEL_DIR = YOLO_DIR / "labels"
YOLO_IMAGE_TRAIN_DIR = YOLO_IMAGE_DIR / "train"
YOLO_IMAGE_VAL_DIR = YOLO_IMAGE_DIR / "val"
YOLO_LABEL_TRAIN_DIR = YOLO_LABEL_DIR / "train"
YOLO_LABEL_VAL_DIR = YOLO_LABEL_DIR / "val"

yolo_dir = {
  "root": YOLO_DIR,
  "train": {
    "images": YOLO_IMAGE_TRAIN_DIR,
    "labels": YOLO_LABEL_TRAIN_DIR
  },
  "val": {
    "images": YOLO_IMAGE_VAL_DIR,
    "labels": YOLO_LABEL_VAL_DIR
  }
}

In [70]:
YOLO_IMAGE_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
YOLO_IMAGE_VAL_DIR.mkdir(parents=True, exist_ok=True)
YOLO_LABEL_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
YOLO_LABEL_VAL_DIR.mkdir(parents=True, exist_ok=True)

In [71]:
annotaiton_meta["annotations"][0]

{'id': 1,
 'image_id': 0,
 'category_id': 6,
 'segmentation': [[561.0,
   1238.0,
   568.0,
   1201.0,
   567.0,
   1175.0,
   549.0,
   1127.0,
   538.0,
   1089.0,
   519.0,
   1043.0,
   517.0,
   1005.0,
   523.0,
   964.0,
   529.0,
   945.0,
   520.0,
   896.0,
   525.0,
   862.0,
   536.0,
   821.0,
   554.0,
   769.0,
   577.0,
   727.0,
   595.0,
   678.0,
   596.0,
   585.0,
   588.0,
   346.0,
   581.0,
   328.0,
   569.0,
   306.0,
   570.0,
   276.0,
   576.0,
   224.0,
   560.0,
   205.0,
   564.0,
   170.0,
   578.0,
   154.0,
   608.0,
   136.0,
   649.0,
   127.0,
   688.0,
   127.0,
   726.0,
   129.0,
   759.0,
   141.0,
   784.0,
   153.0,
   792.0,
   177.0,
   788.0,
   193.0,
   782.0,
   209.0,
   792.0,
   238.0,
   802.0,
   271.0,
   802.0,
   294.0,
   791.0,
   319.0,
   789.0,
   360.0,
   794.0,
   395.0,
   810.0,
   529.0,
   819.0,
   609.0,
   841.0,
   675.0,
   882.0,
   728.0,
   916.0,
   781.0,
   928.0,
   802.0,
   938.0,
   834.0,
   940.0,
  

In [72]:
annotaiton_meta.keys()

dict_keys(['info', 'images', 'annotations', 'scene_annotations', 'licenses', 'categories', 'scene_categories'])

In [73]:
annotaiton_meta["images"][0]

{'id': 0,
 'width': 1537,
 'height': 2049,
 'file_name': 'batch_1/000006.jpg',
 'license': None,
 'flickr_url': 'https://img.tacodataset.org/img/33978196618.jpg',
 'coco_url': None,
 'date_captured': None,
 'flickr_640_url': 'https://img.tacodataset.org/thumb/33978196618_640.jpg'}

In [74]:
annotations_by_image_id = {}

for annotation in annotaiton_meta["annotations"]:
  annotations_by_image_id.setdefault(annotation["image_id"], []).append(
    (annotation["category_id"], annotation["bbox"])
  )

from collections import Counter
from random import Random

# 카테고리 list[이미지 id당 등장하는 set[category]]
image_classes = [
  {cat for cat, _ in annotations_by_image_id.get(image["id"], [])}
  for image in annotaiton_meta["images"]
]

# 전체에 대한 클래스별 카운터 -> class_id: count
class_counts = Counter(cat for classes in image_classes for cat in classes)

groups = {}
for i, classes in enumerate(image_classes):
  # shortcut: 여러 클래스가 있는 이미지는 가장 드문 클래스 기준으로 묶는다. 정확한 다중 라벨 층화가 필요하면 교체한다.
  group = min(classes, key=lambda cat: class_counts[cat]) if classes else None
  groups.setdefault(group, []).append(i)

rng = Random(42)
for indices in groups.values():
  rng.shuffle(indices)

val_target = round(len(image_classes) * VAL_RATIO)
val_per_group = {cat: int(len(indices) * VAL_RATIO) for cat, indices in groups.items()}
remaining = val_target - sum(val_per_group.values())
for cat in sorted(groups, key=lambda cat: len(groups[cat]) * VAL_RATIO % 1, reverse=True)[:remaining]:
  val_per_group[cat] += 1

val_indices = []
train_indices = []
for cat, indices in groups.items():
  val_indices.extend(indices[:val_per_group[cat]])
  train_indices.extend(indices[val_per_group[cat]:])

  
assert len(val_indices) == val_target and len(val_indices) + len(train_indices) == len(image_classes)
assert not set(val_indices) & set(train_indices)

In [75]:
for i, annotation in annotations_by_image_id.items():
  if len(annotation) > 1:
    print(annotation)
    break

[(18, [1.0, 457.0, 1429.0, 1519.0]), (14, [531.0, 292.0, 1006.0, 672.0])]


In [76]:
import shutil
from tqdm import tqdm

# 이전 변환 결과만 지우고 같은 폴더에 다시 생성한다.
for split in ("train", "val"):
  for directory in yolo_dir[split].values():
    for old_file in directory.iterdir():
      if old_file.is_file():
        old_file.unlink()

for split, indices in (("val", val_indices), ("train", train_indices)):
  for i in tqdm(indices, desc=f"{split} extracting"):
    image = annotaiton_meta["images"][i]
    fp = Path(image["file_name"])
    image_path = annotation_path.parent / fp
    output_stem = f"{fp.parent.name}_{fp.stem}"
    shutil.copy2(image_path, yolo_dir[split]["images"] / (output_stem + fp.suffix))

    with open(yolo_dir[split]["labels"] / (output_stem + ".txt"), "w", encoding="utf-8") as f:
      for cat, bbox in annotations_by_image_id.get(image["id"], []):
        x_center = (bbox[0] + bbox[2] / 2) / image["width"]
        y_center = (bbox[1] + bbox[3] / 2) / image["height"]
        f.write(f"{cat} {x_center} {y_center} {bbox[2] / image['width']} {bbox[3] / image['height']}\n")

train extracting: 100%|██████████| 1200/1200 [00:04<00:00, 242.28it/s]


In [77]:
yaml_text = (
    f"path: {YOLO_DIR.resolve().as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
)

for category in annotaiton_meta["categories"]:
    yaml_text += f"  {category['id']}: {json.dumps(category['name'], ensure_ascii=False)}\n"

(YOLO_DIR / "data.yaml").write_text(yaml_text, encoding="utf-8")

1476